# Video Mamba

<div>
    <img src="../images/VIDEOMAMBA.png" width="600">
    <img src="../images/VIDEOMAMBA_SCAN.png" width="400">
</div>

## VideoMamba Architecture

VideoMamba follows the vanilla ViT pipeline but replaces the Transformer encoder with **Bidirectional Mamba (B-Mamba) blocks**, enabling efficient long-range video modeling with **linear complexity**.

### Pipeline

```text
Input Video
      │
      ▼
3D Patch Embedding (1×16×16 Conv)
      │
      ▼
+ [CLS] Token
+ Spatial Position Embedding
+ Temporal Position Embedding
      │
      ▼
L × Bidirectional Mamba Blocks
      │
      ▼
LayerNorm
      │
      ▼
Classification Head
```

- **3D Patch Embedding:** Splits the video into non-overlapping spatiotemporal patches.
- **Position Embeddings:** Spatial and temporal embeddings preserve token order, which is crucial for Mamba.
- **Bidirectional Mamba:** Processes the sequence in both forward and backward directions, allowing each token to access past and future context.
- **Classification:** The final **[CLS]** representation is used for video classification.

---

## Spatiotemporal Scan Strategies

Since Mamba operates on **1D sequences**, video tokens must be serialized before processing.

- **Spatial-First:** Scans all spatial patches within each frame before moving to the next frame. **Best performance** according to the paper.
- **Temporal-First:** Scans tokens across frames for each spatial location, emphasizing temporal continuity.
- **Spatiotemporal v1:** Combines Spatial-First and Temporal-First by assigning half of the layers to each strategy.
- **Spatiotemporal v2:** Executes both scan orders in every layer, providing richer context at approximately **2× computational cost**.

> **Key Idea:** VideoMamba efficiently models videos by converting the 3D spatiotemporal input into a 1D sequence using different scan strategies, while Bidirectional Mamba captures long-range dependencies with linear complexity.

## VideoMamba vs Vision Mamba (ViM)

| Aspect | Vision Mamba (ViM) | VideoMamba |
|--------|---------------------|------------|
| **Input** | 2D images | 3D videos (T × H × W) |
| **Patch Embedding** | 2D patch embedding | 3D patch embedding (1×16×16 Conv) |
| **Position Embedding** | Spatial position embedding | Spatial + Temporal position embeddings |
| **Token Sequence** | Spatial patches only | Spatiotemporal patches |
| **Scanning** | Bidirectional scan over 2D image tokens | Bidirectional scan extended to 3D video tokens |
| **Scan Strategies** | Single spatial scan | Spatial-First, Temporal-First, Spatiotemporal v1/v2 |
| **Encoder** | Bidirectional Mamba blocks | Same B-Mamba blocks adapted for video |
| **Output** | Image classification | Video classification |

### Main Differences

- **VideoMamba extends ViM from 2D images to 3D videos.**
- Uses **3D Patch Embedding** instead of 2D patches.
- Introduces **Temporal Position Embeddings** in addition to spatial embeddings.
- Adapts the **Bidirectional Mamba scan** to process spatiotemporal sequences.
- Proposes multiple **3D scan strategies**, with **Spatial-First** achieving the best performance.
- Maintains **linear computational complexity**, making it efficient for long video sequences.


## 3.4 Masked Modeling

Masked Modeling is a self-supervised pretraining strategy originally introduced in **BERT** and later adapted to vision by **MAE**. Instead of exposing the entire input, part of the data is hidden, forcing the model to learn meaningful representations from the remaining visible tokens.

For example, an image

```text
□ □ □ □
□ □ □ □
□ □ □ □
□ □ □ □
```

can be masked as

```text
□ ■ □ □
■ □ □ ■
□ □ ■ □
□ ■ □ □
```

where **■** represents hidden patches. The model must infer the missing information using only the visible context. In videos, the same idea extends across both **space and time**, encouraging the model to capture fine-grained temporal relationships.

---

### VideoMamba Pretraining

Following the idea of **UMT**, VideoMamba does **feature alignment** instead of reconstructing pixels.

During the first stage, VideoMamba is trained from scratch using only video data. The same video is processed by both **VideoMamba (student)** and a pretrained **CLIP-ViT (teacher)**, and the student learns to match the teacher's final feature representation.

```text
Video
      │
      ├──────────────► CLIP-ViT (Teacher)
      │                     │
      │                     ▼
      │                 Feature
      │
      ▼
VideoMamba (Student)
      │
      ▼
Feature

Loss(Student Feature, Teacher Feature)
```

Only the **visible (unmasked) tokens** are aligned with the teacher features.

After video-only pretraining, a **text encoder** and a **cross-modal decoder (BERT)** are added, allowing VideoMamba to continue pretraining on **image-text** and **video-text** datasets.

```text
Video
      │
      ▼
VideoMamba
      │
      ├──────────────┐
      │              │
Text Encoder         │
      │              │
      └──────► BERT Decoder
```

---

### Difference from UMT

UMT performs **multi-layer feature alignment**, matching intermediate representations between the student and teacher.

```text
Teacher Layer1 ◄──► Student Layer1
Teacher Layer2 ◄──► Student Layer2
Teacher Layer3 ◄──► Student Layer3
Teacher Layer4 ◄──► Student Layer4
```

VideoMamba instead aligns **only the final output features**, since its SSM-based architecture differs significantly from the Transformer's internal representations.

```text
Teacher Final Feature
          ▲
          │
Student Final Feature
```

---

## Masking Strategies

Unlike Transformers, each Mamba block begins with a **1D convolution**, which benefits from **continuous token sequences**. Randomly masking patches often breaks this continuity, reducing the effectiveness of local feature extraction.

Random masking produces fragmented visible tokens:

```text
Visible Hidden Hidden Visible Hidden Visible Hidden Hidden Visible
```

while row-based masking preserves longer continuous regions:

```text
Visible Visible Visible Visible Hidden Hidden Hidden Hidden
```

or spatially

```text
████████
████████
........
........
```

To better exploit this property, VideoMamba evaluates several masking strategies (Fig. 5).

### (a) Input Video

Original video without masking.

### (b) Random Masking

Random patches are independently removed.

- Standard MAE strategy.
- Breaks token continuity.

### (c) Tube Masking

The same spatial patch is masked across consecutive frames.

- Preserves temporal consistency.
- Commonly used in VideoMAE.

### (d) Clip-Row Masking

Entire rows of patches are masked across the **entire video clip**.

- Produces long continuous visible token sequences.
- Better matches Mamba's sequential processing.

### (e) Frame-Row Masking

Rows are masked independently in each frame.

- Similar to Clip-Row.
- Different rows may be hidden in different frames.

### (f) Attention Masking

Masks are generated according to token importance rather than randomly.

- Preserves meaningful neighboring tokens.
- Better exploits the local modeling capability of Mamba's 1D convolution.

> **Key Idea:** Since Mamba processes sequences through a **1D convolution before the SSM**, masking strategies that preserve **continuous visible tokens** (especially row masking) provide better representations than purely random masking.

<div>
    <img src="../images/VIDEOMAMBA_MASKING.png" width="800">
</div>

## Results

On **ImageNet-1K**, **VideoMamba-M** outperformed other isotropic architectures while using fewer parameters, achieving **+0.8%** higher top-1 accuracy than **ConvNeXt-B** and **+2.0%** over **DeiT-B**. By increasing the input resolution, it further reached **84.0% top-1 accuracy** with only **74M parameters**, demonstrating strong scalability and providing better representations that also benefit downstream video tasks.

<div>
    <img src="../images/VIDEOMAMBA_T2.png" width="800">
</div>

### Results (Table 6)

Table 6 compares VideoMamba with previous methods on the **Breakfast (BF)** and **COIN** long-term video understanding benchmarks. VideoMamba consistently achieves state-of-the-art or competitive performance while being trained **end-to-end**, without relying on pre-extracted features. Performance generally improves with **larger model sizes (Ti → S → M)**, **more input frames (32 → 64)**, and **masked pretraining (†)**. The best results are obtained by **VideoMamba-M†**, reaching **97.9% Top-1** on Breakfast and **90.4% Top-1** on COIN, demonstrating the effectiveness of VideoMamba for long-duration video understanding.

<div>
    <img src="../images/VIDEOMAMBA_T6.png" width="800">
</div>

| Aspect | Action Recognition (e.g., Kinetics-400) | Long-Term Video Understanding (e.g., Breakfast, COIN) |
|--------|------------------------------------------|--------------------------------------------------------|
| **Video Length** | Short (≈5–10 s) | Long (minutes) |
| **Content** | Usually a single action | Multiple sequential actions |
| **Temporal Dependency** | Low to moderate | High |
| **Objective** | Recognize one action | Understand an entire activity/procedure |
| **Example Video** | Person clapping | Preparing breakfast, repairing a bicycle |
| **Example Sequence** | *Clapping* | Take cup → Pour coffee → Stir → Drink |
| **Model Input** | Short video clip | Complete long video |
| **Model Output** | One action label | One activity/procedure label |
| **Evaluation Metric** | Top-1 / Top-5 Accuracy | Top-1 Accuracy |
| **Representative Datasets** | Kinetics-400, Kinetics-600, UCF101, HMDB51 | Breakfast, COIN, LVU |

### Examples

**Action Recognition (Kinetics)**

```text
Input:
Person jumps into a swimming pool.

Output:
"Diving"
```

**Long-Term Video Understanding (Breakfast / COIN)**

```text
Input:
Take bread
→ Toast bread
→ Fry egg
→ Assemble sandwich
→ Serve

Output:
"Prepare Breakfast"
```

> **Key Difference:** Action recognition focuses on identifying **a single action** in a short clip, whereas long-term video understanding requires modeling **multiple actions and their temporal relationships** to recognize the overall activity or procedure.

> **Multimodal Setting:** Similar to **CLIP**, VideoMamba learns a **shared embedding space** between **videos** and **text**, replacing CLIP's image encoder with a **VideoMamba encoder**. During inference, a text query is matched against video embeddings to retrieve the most relevant videos (**zero-shot text-to-video retrieval**).

### Results (Table 8)

Table 8 reports **zero-shot text-to-video retrieval** performance on five benchmarks (MSRVTT, DiDeMo, ActivityNet, LSMDC, and MSVD), evaluated using **Recall@1, @5, and @10**. Under the same pretraining data and training protocol as UMT, **VideoMamba consistently matches or outperforms ViT-based methods**, with particularly notable improvements on **longer and more complex video datasets** such as **ActivityNet, DiDeMo, and LSMDC**. These results demonstrate that VideoMamba effectively learns **cross-modal (video-text) representations**, making it well-suited for multimodal video understanding and retrieval tasks.